<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 从头开始​​的字节对编码 (BPE) 分词器

- 这是一个独立的笔记本，实现了流行的字节对编码 (BPE) 标记化算法，该算法用于 GPT-2 到 GPT-4、Llama 3 等模型，从头开始用于教育目的
- 有关代币化目的的更多详细信息，请参阅 [第 2 章](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/ch02.ipynb)；这里的代码是解释 BPE 算法的额外材料
- OpenAI 为训练原始 GPT 模型而实现的原始 BPE 标记器可以在此处找到 [](https://github.com/openai/gpt-2/blob/master/src/encoder.py)
- BPE 算法最初于 1994 年描述：“[数据压缩的新算法](https://github.com/tpn/pdfs/blob/master/A%20New%20Algorithm%20for%20Data%20Compression%20(1994).pdf)”，作者：Philip Gage
- 由于其计算性能，现在包括 Llama 3 在内的大多数项目都使用 OpenAI 的开源 [tiktoken 库](https://github.com/openai/tiktoken)；例如，它允许加载预训练的 GPT-2 和 GPT-4 分词器（Llama 3 模型也使用 GPT-4 分词器进行训练）
- 上面的实现与我在本笔记本中的实现之间的区别在于，它还包括用于训练分词器的功能（用于教育目的）
- 还有一个名为 [minBPE](https://github.com/karpathy/minbpe) 的实现，带有培训支持，可能性能更高（我在这里的实现主要用于教育目的）；与 `minbpe` 相比，我的实现还允许加载原始 OpenAI 分词器词汇和 BPE“合并”（此外，Hugging Face 分词器还能够训练和加载各种分词器；有关更多信息，请参阅 [GitHub 讨论](https://github.com/rasbt/LLMs-from-scratch/discussions/485)，该读者曾在尼泊尔语言上训练过 BPE 分词器）

 
# 1. 字节对编码（BPE）背后的主要思想

- BPE 的主要思想是将文本转换为整数表示（令牌 ID）以进行 LLM 训练（请参阅 [第 2 章](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/ch02.ipynb)）

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/bpe-overview.webp" width="600px">

 
## 1.1 位和字节

- 在介绍 BPE 算法之前，我们先介绍一下字节的概念
- 考虑将文本转换为字节数组（BPE 毕竟代表“字节”对编码）：

In [1]:
text = "This is some text"
byte_ary = bytearray(text, "utf-8")
print(byte_ary)

bytearray(b'This is some text')


- 当我们在 `bytearray` 对象上调用 `list()` 时，每个字节都被视为单个元素，结果是与字节值对应的整数列表：

In [2]:
ids = list(byte_ary)
print(ids)

[84, 104, 105, 115, 32, 105, 115, 32, 115, 111, 109, 101, 32, 116, 101, 120, 116]


- 这是将文本转换为 LLM 嵌入层所需的令牌 ID 表示的有效方法
- 然而，这种方法的缺点是它为每个字符创建一个 ID（对于短文本来说有很多 ID！）
- 即，这意味着对于 17 个字符的输入文本，我们必须使用 17 个令牌 ID 作为 LLM 的输入：

In [3]:
print("Number of characters:", len(text))
print("Number of token IDs:", len(ids))

Number of characters: 17
Number of token IDs: 17


- 如果您以前使用过大语言模型，您可能知道 BPE 分词器有一个词汇表，其中我们有整个单词或子词而不是每个字符的标记 ID
- 例如，GPT-2 标记器将相同的文本（“这是一些文本”）标记为仅 4 个标记，而不是 17 个标记：`1212, 318, 617, 2420`
- 您可以使用交互式 [tiktoken 应用程序](https://tiktokenizer.vercel.app/?model=gpt2) 或 [tiktoken 库](https://github.com/openai/tiktoken) 仔细检查这一点：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/tiktokenizer.webp" width="600px">

```python
import tiktoken

gpt2_tokenizer = tiktoken.get_encoding("gpt2")
gpt2_tokenizer.encode("This is some text")
# prints [1212, 318, 617, 2420]
```

- 由于一个字节由 8 位组成，因此单个字节可以表示 2<sup>8</sup> = 256 个可能的值，范围从 0 到 255
- 您可以通过执行代码 `bytearray(range(0, 257))` 来确认这一点，该代码会警告您 `ValueError: byte must be in range(0, 256)`)
- BPE 标记器通常使用这 256 个值作为其前 256 个单字符标记；人们可以通过运行以下代码来直观地检查这一点：

```python
import tiktoken
gpt2_tokenizer = tiktoken.get_encoding("gpt2")

for i in range(300):
    decoded = gpt2_tokenizer.decode([i])
    print(f"{i}: {decoded}")
"""
prints:
0: !
1: "
2: #
...
255: �  # <---- single character tokens up to here
256:  t
257:  a
...
298: ent
299:  n
"""
```

- 上面，请注意，条目 256 和 257 不是单字符值，而是双字符值（一个空格 + 一个字母），这是原始 GPT-2 BPE 分词器的一个小缺点（这在 GPT-4 分词器中已得到改进）

 
## 1.2 建立词汇表

- BPE 标记化算法的目标是构建常见子词的词汇表，例如 `298: ent`（例如，可以在 *entangle、entertainment、enter、entry、entity、...* 中找到），甚至是完整的单词，例如 

```
318: is
617: some
1212: This
2420: text
```

- BPE 算法最初于 1994 年描述：“[数据压缩的新算法](https://github.com/tpn/pdfs/blob/master/A%20New%20Algorithm%20for%20Data%20Compression%20(1994).pdf)”，作者：Philip Gage
- 在我们开始实际的代码实现之前，今天用于 LLM 分词器的形式可以总结为以下几节所述。

 
## 1.3 BPE算法概述

**1.识别频繁配对**
- 在每次迭代中，扫描文本以查找最常出现的字节（或字符）对

**2.替换并记录**

- 用新的占位符 ID 替换该对（尚未使用的占位符 ID，例如，如果我们以 0...255 开头，则第一个占位符将为 256）
- 在查找表中记录此映射
- 查找表的大小是一个超参数，也称为“词汇大小”（对于 GPT-2，即
50,257)

**3.重复直到没有收益**

- 不断重复步骤1和2，不断合并最频繁的对
- 当无法进一步压缩时停止（例如，没有配对出现超过一次）

**解压（解码）**

- 要恢复原始文本，请使用查找表将每个 ID 替换为其对应的 ID 对，从而反转该过程

 
## 1.4 BPE算法示例

### 1.4.1 编码部分的具体示例（1.3节中的步骤1和2）

- 假设我们有文本（训练数据集）`the cat in the hat`，我们想从中构建 BPE 分词器的词汇表

**迭代 1**

1. 识别频繁配对
  - 在本文中，“th”出现两次（在开头和第二个“e”之前）

2. 替换并记录
  - 将“th”替换为尚未使用的新令牌 ID，例如 256
  - 新文本为：`<256>e cat in <256>e hat`
  - 新词汇是

```
  0: ...
  ...
  256: "th"
```

**迭代2**

1. **识别频繁配对**  
   - 在文本`<256>e cat in <256>e hat`中，`<256>e`对出现了两次

2. **替换并记录**  
   - 将 `<256>e` 替换为尚未使用的新令牌 ID，例如 `257`。  
   - 新文本是：
     ```
     <257> cat in <257> hat
     ```
   - 更新后的词汇为：
     ```
     0: ...
     ...
     256: "th"
     257: "<256>e"
     ```

**迭代3**

1. **识别频繁配对**  
   - 在文本 `<257> cat in <257> hat` 中，`<257> ` 对出现两次（一次在开头，一次在“帽子”之前）。

2. **替换并记录**  
   - 将 `<257> ` 替换为尚未使用的新令牌 ID，例如 `258`。  
   - 新文本是：
     ```
     <258>cat in <258>hat
     ```
   - 更新后的词汇为：
     ```
     0: ...
     ...
     256: "th"
     257: "<256>e"
     258: "<257> "
     ```
     
- 等等

 
### 1.4.2 解码部分的具体示例（1.3节中的步骤3）

- 为了恢复原始文本，我们通过按照引入的相反顺序将每个令牌 ID 替换为其相应的对来反转该过程
- 从最终的压缩文本开始：`<258>cat in <258>hat`
- 替代 `<258>` → `<257> `: `<257> cat in <257> hat`  
- 替代 `<257>` → `<256>e`: `<256>e cat in <256>e hat`
- 替换 `<256>` →“第”：`the cat in the hat`

 
## 2. 一个简单的 BPE 实现

- 下面是上述算法的实现，作为模仿 `tiktoken` Python 用户界面的 Python 类
- 请注意，上面的编码部分通过 `train()` 描述了原始训练步骤；然而，`encode()` 方法的工作原理类似（尽管由于特殊的令牌处理，它看起来有点复杂）：

1. 将输入文本分割成单独的字节
2. 当相邻标记（对）与学习的 BPE 合并中的任何对匹配时，重复查找并替换（合并）相邻标记（对）（从最高到最低“排名”，即按照学习的顺序）
3. 继续合并，直到无法再应用合并为止
4. 最终的token ID列表就是编码后的输出

In [4]:
from collections import Counter, deque
from functools import lru_cache
import re
import json


class BPETokenizerSimple:
    def __init__(self):
        # Maps token_id to token_str (e.g., {11246: "some"})
        self.vocab = {}
        # Maps token_str to token_id (e.g., {"some": 11246})
        self.inverse_vocab = {}
        # Dictionary of BPE merges: {(token_id1, token_id2): merged_token_id}
        self.bpe_merges = {}

        # For the official OpenAI GPT-2 merges, use a rank dict:
        #  of form {(string_A, string_B): rank}, where lower rank = higher priority
        self.bpe_ranks = {}

    def train(self, text, vocab_size, allowed_special={"<|endoftext|>"}):
        """
        Train the BPE tokenizer from scratch.

        Args:
            text (str): The training text.
            vocab_size (int): The desired vocabulary size.
            allowed_special (set): A set of special tokens to include.
        """

        # Pre-tokenize training text using the same boundary rules as encode()
        tokens = self.pretokenize_text(text)

        # Initialize vocab with unique characters, including "Ġ" if present
        # Start with the first 256 ASCII characters
        unique_chars = [chr(i) for i in range(256)]
        unique_chars.extend(
            char for char in sorted({char for token in tokens for char in token})
            if char not in unique_chars
        )
        if "Ġ" not in unique_chars:
            unique_chars.append("Ġ")

        self.vocab = {i: char for i, char in enumerate(unique_chars)}
        self.inverse_vocab = {char: i for i, char in self.vocab.items()}

        # Add allowed special tokens
        if allowed_special:
            for token in allowed_special:
                if token not in self.inverse_vocab:
                    new_id = len(self.vocab)
                    self.vocab[new_id] = token
                    self.inverse_vocab[token] = new_id

        # Tokenize each pre-token into character IDs
        token_id_sequences = [
            [self.inverse_vocab[char] for char in token]
            for token in tokens
        ]

        # BPE steps 1-3: Repeatedly find and replace frequent pairs
        for new_id in range(len(self.vocab), vocab_size):
            pair_id = self.find_freq_pair(token_id_sequences, mode="most")
            if pair_id is None:
                break
            token_id_sequences = self.replace_pair(token_id_sequences, pair_id, new_id)
            self.bpe_merges[pair_id] = new_id

        # Build the vocabulary with merged tokens
        for (p0, p1), new_id in self.bpe_merges.items():
            merged_token = self.vocab[p0] + self.vocab[p1]
            self.vocab[new_id] = merged_token
            self.inverse_vocab[merged_token] = new_id

    def load_vocab_and_merges_from_openai(self, vocab_path, bpe_merges_path):
        """
        Load pre-trained vocabulary and BPE merges from OpenAI's GPT-2 files.

        Args:
            vocab_path (str): Path to the vocab file (GPT-2 calls it 'encoder.json').
            bpe_merges_path (str): Path to the bpe_merges file  (GPT-2 calls it 'vocab.bpe').
        """
        # Load vocabulary
        with open(vocab_path, "r", encoding="utf-8") as file:
            loaded_vocab = json.load(file)
            # encoder.json is {token_str: id}; we want id->str and str->id
            self.vocab = {int(v): k for k, v in loaded_vocab.items()}
            self.inverse_vocab = {k: int(v) for k, v in loaded_vocab.items()}
    
        # Must have GPT-2's printable newline character 'Ċ' (U+010A) at id 198
        if "Ċ" not in self.inverse_vocab or self.inverse_vocab["Ċ"] != 198:
            raise KeyError("Vocabulary missing GPT-2 newline glyph 'Ċ' at id 198.")
    
        # Must have <|endoftext|> at 50256
        if "<|endoftext|>" not in self.inverse_vocab or self.inverse_vocab["<|endoftext|>"] != 50256:
            raise KeyError("Vocabulary missing <|endoftext|> at id 50256.")
    
        # Provide a convenience alias for '\n' -> 198
        # Keep printable character 'Ċ' in vocab so BPE merges keep working
        if "\n" not in self.inverse_vocab:
            self.inverse_vocab["\n"] = self.inverse_vocab["Ċ"]

        if "\r" not in self.inverse_vocab:
            if 201 in self.vocab:
                self.inverse_vocab["\r"] = 201
            else:
                raise KeyError("Vocabulary missing carriage return token at id 201.")

        # Load GPT-2 merges and store ranks
        self.bpe_ranks = {}
        with open(bpe_merges_path, "r", encoding="utf-8") as file:
            lines = file.readlines()
            if lines and lines[0].startswith("#"):
                lines = lines[1:]
    
            rank = 0
            for line in lines:
                token1, *rest = line.strip().split()
                if len(rest) != 1:
                    continue
                token2 = rest[0]
                if token1 in self.inverse_vocab and token2 in self.inverse_vocab:
                    self.bpe_ranks[(token1, token2)] = rank
                    rank += 1
                else:
                    # Safe to skip pairs whose symbols are not in vocab
                    pass


    def encode(self, text, allowed_special=None):
        """
        Encode the input text into a list of token IDs, with tiktoken-style handling of special tokens.
    
        Args:
            text (str): The input text to encode.
            allowed_special (set or None): Special tokens to allow passthrough. If None, special handling is disabled.
    
        Returns:
            List of token IDs.
        """
    
        # ---- This section is to mimic tiktoken in terms of allowed special tokens ----
        specials_in_vocab = [
            tok for tok in self.inverse_vocab
            if tok.startswith("<|") and tok.endswith("|>")
        ]
        if allowed_special is None:
            # Nothing is allowed
            disallowed = [tok for tok in specials_in_vocab if tok in text]
            if disallowed:
                raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")
        else:
            # Some spefic tokens are allowed (e.g., we use this for <|endoftext|>)
            disallowed = [tok for tok in specials_in_vocab if tok in text and tok not in allowed_special]
            if disallowed:
                raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")
        # -----------------------------------------------------------------------------

        token_ids = []
        # If some specials are allowed, split around them and passthrough those ids
        if allowed_special is not None and len(allowed_special) > 0:
            special_pattern = "(" + "|".join(
                re.escape(tok) for tok in sorted(allowed_special, key=len, reverse=True)
            ) + ")"
    
            last_index = 0
            for match in re.finditer(special_pattern, text):
                prefix = text[last_index:match.start()]
                token_ids.extend(self.encode(prefix, allowed_special=None))  # encode prefix normally
    
                special_token = match.group(0)
                if special_token in self.inverse_vocab:
                    token_ids.append(self.inverse_vocab[special_token])
                else:
                    raise ValueError(f"Special token {special_token} not found in vocabulary.")
                last_index = match.end()
    
            text = text[last_index:]  # remainder to process normally
    
            # Extra guard for any other special literals left over
            disallowed = [
                tok for tok in self.inverse_vocab
                if tok.startswith("<|") and tok.endswith("|>") and tok in text and tok not in allowed_special
            ]
            if disallowed:
                raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")

    
        # ---- Newline and carriage return handling ----
        tokens = self.pretokenize_text(text)
        # ---------------------------------------------------------------
    
        # Map tokens -> ids (BPE if needed)
        for tok in tokens:
            if tok in self.inverse_vocab:
                token_ids.append(self.inverse_vocab[tok])
            else:
                token_ids.extend(self.tokenize_with_bpe(tok))
    
        return token_ids

    def tokenize_with_bpe(self, token):
        """
        Tokenize a single token using BPE merges.

        Args:
            token (str): The token to tokenize.

        Returns:
            List[int]: The list of token IDs after applying BPE.
        """
        # Tokenize the token into individual characters (as initial token IDs)
        token_ids = [self.inverse_vocab.get(char, None) for char in token]
        if None in token_ids:
            missing_chars = [char for char, tid in zip(token, token_ids) if tid is None]
            raise ValueError(f"Characters not found in vocab: {missing_chars}")

        # If we haven't loaded OpenAI's GPT-2 merges, use my approach
        if not self.bpe_ranks:
            can_merge = True
            while can_merge and len(token_ids) > 1:
                can_merge = False
                new_tokens = []
                i = 0
                while i < len(token_ids) - 1:
                    pair = (token_ids[i], token_ids[i + 1])
                    if pair in self.bpe_merges:
                        merged_token_id = self.bpe_merges[pair]
                        new_tokens.append(merged_token_id)
                        # Uncomment for educational purposes:
                        # print(f"Merged pair {pair} -> {merged_token_id} ('{self.vocab[merged_token_id]}')")
                        i += 2  # Skip the next token as it's merged
                        can_merge = True
                    else:
                        new_tokens.append(token_ids[i])
                        i += 1
                if i < len(token_ids):
                    new_tokens.append(token_ids[i])
                token_ids = new_tokens
            return token_ids

        # Otherwise, do GPT-2-style merging with the ranks:
        # 1) Convert token_ids back to string "symbols" for each ID
        symbols = [self.vocab[id_num] for id_num in token_ids]

        # Repeatedly merge all occurrences of the lowest-rank pair
        while True:
            # Collect all adjacent pairs
            pairs = set(zip(symbols, symbols[1:]))
            if not pairs:
                break

            # Find the pair with the best (lowest) rank
            min_rank = float("inf")
            bigram = None
            for p in pairs:
                r = self.bpe_ranks.get(p, float("inf"))
                if r < min_rank:
                    min_rank = r
                    bigram = p

            # If no valid ranked pair is present, we're done
            if bigram is None or bigram not in self.bpe_ranks:
                break

            # Merge all occurrences of that pair
            first, second = bigram
            new_symbols = []
            i = 0
            while i < len(symbols):
                # If we see (first, second) at position i, merge them
                if i < len(symbols) - 1 and symbols[i] == first and symbols[i+1] == second:
                    new_symbols.append(first + second)  # merged symbol
                    i += 2
                else:
                    new_symbols.append(symbols[i])
                    i += 1
            symbols = new_symbols

            if len(symbols) == 1:
                break

        # Finally, convert merged symbols back to IDs
        merged_ids = [self.inverse_vocab[sym] for sym in symbols]
        return merged_ids

    def decode(self, token_ids):
        """
        Decode a list of token IDs back into a string.

        Args:
            token_ids (List[int]): The list of token IDs to decode.

        Returns:
            str: The decoded string.
        """
        out = []
        for tid in token_ids:
            if tid not in self.vocab:
                raise ValueError(f"Token ID {tid} not found in vocab.")
            tok = self.vocab[tid]

            # Map GPT-2 special chars back to real chars
            if tid == 198 or tok == "\n":
                out.append("\n")
            elif tid == 201 or tok == "\r":
                out.append("\r")
            elif tok.startswith("Ġ"):
                out.append(" " + tok[1:])
            else:
                out.append(tok)
        return "".join(out)

    def save_vocab_and_merges(self, vocab_path, bpe_merges_path):
        """
        Save the vocabulary and BPE merges to JSON files.

        Args:
            vocab_path (str): Path to save the vocabulary.
            bpe_merges_path (str): Path to save the BPE merges.
        """
        # Save vocabulary
        with open(vocab_path, "w", encoding="utf-8") as file:
            json.dump(self.vocab, file, ensure_ascii=False, indent=2)

        # Save BPE merges as a list of dictionaries
        with open(bpe_merges_path, "w", encoding="utf-8") as file:
            merges_list = [{"pair": list(pair), "new_id": new_id}
                           for pair, new_id in self.bpe_merges.items()]
            json.dump(merges_list, file, ensure_ascii=False, indent=2)

    def load_vocab_and_merges(self, vocab_path, bpe_merges_path):
        """
        Load the vocabulary and BPE merges from JSON files.

        Args:
            vocab_path (str): Path to the vocabulary file.
            bpe_merges_path (str): Path to the BPE merges file.
        """
        # Load vocabulary
        with open(vocab_path, "r", encoding="utf-8") as file:
            loaded_vocab = json.load(file)
            self.vocab = {int(k): v for k, v in loaded_vocab.items()}
            self.inverse_vocab = {v: int(k) for k, v in loaded_vocab.items()}

        # Load BPE merges
        with open(bpe_merges_path, "r", encoding="utf-8") as file:
            merges_list = json.load(file)
            for merge in merges_list:
                pair = tuple(merge["pair"])
                new_id = merge["new_id"]
                self.bpe_merges[pair] = new_id

    @lru_cache(maxsize=None)
    def get_special_token_id(self, token):
        return self.inverse_vocab.get(token, None)

    @staticmethod
    def pretokenize_text(text):
        tokens = []
        parts = re.split(r'(\r\n|\r|\n)', text)
        for part in parts:
            if part == "":
                continue
            if part == "\r\n":
                tokens.append("\r")
                tokens.append("\n")
                continue
            if part == "\r":
                tokens.append("\r")
                continue
            if part == "\n":
                tokens.append("\n")
                continue

            # Normal chunk without line breaks:
            # - If spaces precede a word, prefix the first word with 'Ġ' and
            #   add standalone 'Ġ' for additional spaces
            # - If spaces trail the chunk (e.g., before a newline) add
            #   standalone 'Ġ' tokens (tiktoken produces id 220 for 'Ġ')
            pending_spaces = 0
            for m in re.finditer(r'( +)|(\S+)', part):
                if m.group(1) is not None:
                    pending_spaces += len(m.group(1))
                else:
                    word = m.group(2)
                    if pending_spaces > 0:
                        for _ in range(pending_spaces - 1):
                            tokens.append("Ġ")  # remaining spaces as standalone
                        tokens.append("Ġ" + word) # one leading space
                        pending_spaces = 0
                    else:
                        tokens.append(word)
            # Trailing spaces (no following word): add standalone 'Ġ' tokens
            for _ in range(pending_spaces):
                tokens.append("Ġ")
        return tokens

    @staticmethod
    def find_freq_pair(token_id_sequences, mode="most"):
        pairs = Counter(
            pair
            for token_ids in token_id_sequences
            for pair in zip(token_ids, token_ids[1:])
        )

        if not pairs:
            return None

        if mode == "most":
            return max(pairs.items(), key=lambda x: x[1])[0]
        elif mode == "least":
            return min(pairs.items(), key=lambda x: x[1])[0]
        else:
            raise ValueError("Invalid mode. Choose 'most' or 'least'.")

    @staticmethod
    def replace_pair(token_id_sequences, pair_id, new_id):
        replaced_sequences = []

        for token_ids in token_id_sequences:
            dq = deque(token_ids)
            replaced = []

            while dq:
                current = dq.popleft()
                if dq and (current, dq[0]) == pair_id:
                    replaced.append(new_id)
                    # Remove the 2nd token of the pair, 1st was already removed
                    dq.popleft()
                else:
                    replaced.append(current)

            replaced_sequences.append(replaced)

        return replaced_sequences

- 上面的 `BPETokenizerSimple` 类中有很多代码，详细讨论它超出了本笔记本的范围，但下一节提供了用法的简短概述，以便更好地理解类方法

## 3. BPE 实施演练

- 在实践中，我强烈建议使用 [tiktoken](https://github.com/openai/tiktoken)，因为我上面的实现侧重于可读性和教育目的，而不是性能
- 不过用法或多或少与tiktoken类似，只不过tiktoken没有训练方法
- 让我们通过查看下面的一些示例来了解上面的 `BPETokenizerSimple` Python 代码是如何工作的（详细的代码讨论超出了本笔记本的范围）

### 3.1 训练、编码和解码

- 首先，让我们考虑一些示例文本作为我们的训练数据集：

In [5]:
import os
import requests

def download_file_if_absent(url, filename, search_dirs):
    for directory in search_dirs:
        file_path = os.path.join(directory, filename)
        if os.path.exists(file_path):
            print(f"{filename} already exists in {file_path}")
            return file_path

    target_path = os.path.join(search_dirs[0], filename)
    try:
        response = requests.get(url, stream=True, timeout=60)
        response.raise_for_status()
        with open(target_path, "wb") as out_file:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    out_file.write(chunk)
        print(f"Downloaded {filename} to {target_path}")
    except Exception as e:
        print(f"Failed to download {filename}. Error: {e}")

    return target_path


verdict_path = download_file_if_absent(
    url=(
         "https://raw.githubusercontent.com/rasbt/"
         "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
         "the-verdict.txt"
    ),
    filename="the-verdict.txt",
    search_dirs=["ch02/01_main-chapter-code/", "../01_main-chapter-code/", "."]
)

with open(verdict_path, "r", encoding="utf-8") as f: # added ../01_main-chapter-code/
    text = f.read()

the-verdict.txt already exists in ../01_main-chapter-code/the-verdict.txt


- 接下来，我们初始化并训练词汇量为 1,000 的 BPE 分词器
- 请注意，由于前面讨论的字节值，默认情况下词汇表大小已经为 256，因此我们仅“学习”744 个词汇表条目（如果我们考虑 `<|endoftext|>` 特殊标记和 `Ġ` 空白标记；因此，准确地说是 742 个）
- 作为比较，GPT-2词汇量为50,257个令牌，GPT-4词汇量为100,256个令牌（tiktoken中的`cl100k_base`），GPT-4o使用199,997个令牌（tiktoken中的`o200k_base`）；与我们上面的简单示例文本相比，它们都有更大的训练集

In [6]:
tokenizer = BPETokenizerSimple()
tokenizer.train(text, vocab_size=1000, allowed_special={"<|endoftext|>"})

- 您可能想检查词汇内容（但请注意，它会创建一个很长的列表）

In [7]:
# print(tokenizer.vocab)
print(len(tokenizer.vocab))

1000


- 这个词汇是通过合并742次创建的（`= 1000 - len(range(0, 256)) - len(special_tokens) - "Ġ" = 1000 - 256 - 1 - 1 = 742`）

In [8]:
print(len(tokenizer.bpe_merges))

742


- 这意味着前 256 个条目是单字符标记

- 接下来，让我们使用通过 `encode` 方法创建的合并来编码一些文本：

In [9]:
input_text = "Jack embraced beauty through art and life."
token_ids = tokenizer.encode(input_text)
print(token_ids)

[424, 256, 654, 531, 302, 311, 256, 296, 97, 465, 121, 595, 841, 116, 287, 466, 256, 326, 972, 46]


In [10]:
input_text = "Jack embraced beauty through art and life.<|endoftext|> "
token_ids = tokenizer.encode(input_text, allowed_special={"<|endoftext|>"})
print(token_ids)

[424, 256, 654, 531, 302, 311, 256, 296, 97, 465, 121, 595, 841, 116, 287, 466, 256, 326, 972, 46, 257, 256]


In [11]:
print("Number of characters:", len(input_text))
print("Number of token IDs:", len(token_ids))

Number of characters: 56
Number of token IDs: 22


- 从上面的长度中，我们可以看到一个 42 个字符的句子被编码为 20 个 token ID，与基于字符字节的编码相比，有效地将输入长度大约减少了一半

- 请注意，词汇本身在 `decode()` 方法中使用，它允许我们将令牌 ID 映射回文本：

In [12]:
print(token_ids)

[424, 256, 654, 531, 302, 311, 256, 296, 97, 465, 121, 595, 841, 116, 287, 466, 256, 326, 972, 46, 257, 256]


In [13]:
print(tokenizer.decode(token_ids))

Jack embraced beauty through art and life.<|endoftext|> 


- 迭代每个令牌 ID 可以让我们更好地理解令牌 ID 如何通过词汇进行解码：

In [14]:
for token_id in token_ids:
    print(f"{token_id} -> {tokenizer.decode([token_id])}")

424 -> Jack
256 ->  
654 -> em
531 -> br
302 -> ac
311 -> ed
256 ->  
296 -> be
97 -> a
465 -> ut
121 -> y
595 ->  through
841 ->  ar
116 -> t
287 ->  a
466 -> nd
256 ->  
326 -> li
972 -> fe
46 -> .
257 -> <|endoftext|>
256 ->  


- 正如我们所看到的，大多数令牌 ID 代表 2 个字符的子词；这是因为训练数据文本非常短，没有那么多重复单词，而且我们使用的词汇量相对较小

- 作为总结，调用 `decode(encode())` 应该能够重现任意输入文本：

In [15]:
tokenizer.decode(
    tokenizer.encode("This is some text.")
)

'This is some text.'

In [16]:
tokenizer.decode(
    tokenizer.encode("This is some text with \n newline characters.")
)

'This is some text with \n newline characters.'

### 3.2 保存和加载分词器

- 接下来，让我们看看如何保存经过训练的分词器以供以后重用：

In [17]:
# Save trained tokenizer
tokenizer.save_vocab_and_merges(vocab_path="vocab.json", bpe_merges_path="bpe_merges.txt")

In [18]:
# Load tokenizer
tokenizer2 = BPETokenizerSimple()
tokenizer2.load_vocab_and_merges(vocab_path="vocab.json", bpe_merges_path="bpe_merges.txt")

- 加载的分词器应该能够产生与以前相同的结果：

In [19]:
print(tokenizer2.decode(token_ids))

Jack embraced beauty through art and life.<|endoftext|> 


In [20]:
tokenizer2.decode(
    tokenizer2.encode("This is some text with \n newline characters.")
)

'This is some text with \n newline characters.'

 
### 3.3 从 OpenAI 加载原始 GPT-2 BPE 标记器

- 最后，让我们加载 OpenAI 的 GPT-2 分词器文件

In [21]:
# Download files if not already present in this directory

# Define the directories to search and the files to download
search_directories = ["ch02/02_bonus_bytepair-encoder/gpt2_model/", "../02_bonus_bytepair-encoder/gpt2_model/", "."]

files_to_download = {
    "https://openaipublic.blob.core.windows.net/gpt-2/models/124M/vocab.bpe": "vocab.bpe",
    "https://openaipublic.blob.core.windows.net/gpt-2/models/124M/encoder.json": "encoder.json"
}

# Ensure directories exist and download files if needed
paths = {}
for url, filename in files_to_download.items():
    paths[filename] = download_file_if_absent(url, filename, search_directories)

vocab.bpe already exists in ../02_bonus_bytepair-encoder/gpt2_model/vocab.bpe
encoder.json already exists in ../02_bonus_bytepair-encoder/gpt2_model/encoder.json


- 接下来，我们通过 `load_vocab_and_merges_from_openai` 方法加载文件：

In [22]:
tokenizer_gpt2 = BPETokenizerSimple()
tokenizer_gpt2.load_vocab_and_merges_from_openai(
    vocab_path=paths["encoder.json"], bpe_merges_path=paths["vocab.bpe"]
)

- 词汇量大小应为 `50257`，我们可以通过以下代码确认：

In [23]:
len(tokenizer_gpt2.vocab)

50257

- 我们现在可以通过 `BPETokenizerSimple` 对象使用 GPT-2 分词器：

In [24]:
input_text = "This is some text"
token_ids = tokenizer_gpt2.encode(input_text)
print(token_ids)

[1212, 318, 617, 2420]


In [25]:
print(tokenizer_gpt2.decode(token_ids))

This is some text


- 您可以使用交互式 [tiktoken 应用程序](https://tiktokenizer.vercel.app/?model=gpt2) 或 [tiktoken 库](https://github.com/openai/tiktoken) 仔细检查这会产生正确的令牌：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/tiktokenizer.webp" width="600px">

```python
import tiktoken

gpt2_tokenizer = tiktoken.get_encoding("gpt2")
gpt2_tokenizer.encode("This is some text")
# prints [1212, 318, 617, 2420]
```

 
# 4. 结论

- 就是这样！简而言之，这就是 BPE 的工作原理，包括用于创建新分词器或加载 GPT-2 分词器词汇并与原始 OpenAI GPT-2 模型合并的训练方法
- 我希望您发现这个简短的教程对教育目的有用；如果您有任何疑问，请随时在此处打开新的讨论 [](https://github.com/rasbt/LLMs-from-scratch/discussions/categories/q-a)
- 有关与其他分词器实现的性能比较，请参阅 [此笔记本](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/02_bonus_bytepair-encoder/compare-bpe-tiktoken.ipynb)